A hybrid memory that keeps recent messages word-for-word while summarizing older history.

Think of how you remember a long phone call with a friend. You recall the last few sentences almost word-for-word. That's your short-term memory. The earlier parts? You remember the gist, not the exact words. That's your long-term memory. Summary Buffer Memory works the same way: it keeps recent messages intact and compresses older ones into a summary.

Summary Buffer Memory is a hybrid agent memory technique that combines conversation buffer and summary memory into one system. It keeps recent messages word-for-word while compressing older conversation history into a running LLM-generated summary. This dual-region design fits naturally into customer support agents, long-running chatbots, and any application that needs both precise recent context and historical awareness. The engineering challenge is tuning the transition threshold (the point where messages move from the buffer into the summary) and the token budget split between the two regions.

Summary Buffer Memory works the same way. It keeps a buffer zone of the most recent messages in their original form. Everything older gets compressed into a running summary. The LLM receives both pieces on every call: the summary for historical context, and the raw recent messages for precision.

Pure buffer memory (storing every message) gives exact recall but costs more tokens with each turn. Pure summary memory (compressing everything) saves tokens but loses recent detail. Summary Buffer Memory combines both strengths. It's especially valuable for customer support bots, coaching agents, and project management assistants. These agents need to remember the full arc of a conversation while responding precisely to the latest message.

The core engineering challenge is the transition threshold: the point where messages age out of the buffer and fold into the summary. Set it too high and you waste tokens. Set it too low and you lose important detail.

By the end of this notebook you'll understand:

How to build a summary buffer memory from scratch with the OpenAI SDK.

How incremental summarization works (updating a summary with newly evicted messages).

How to tune the buffer size and observe its effect on token usage.

When this technique wins and when it quietly fails.

### Key Concepts

Buffer zone: The most recent k messages stored word-for-word. These preserve exact wording, nuance, and details for immediate reasoning.

Running summary: A short paragraph that captures key facts and themes from all messages that have aged out of the buffer.

Incremental summarization: Instead of re-summarizing the full history each time, we fold newly evicted messages into the existing summary in a single LLM call.

Transition threshold: The point at which messages age out of the buffer and get folded into the summary. You can trigger it by message count or token count.

Token budget allocation: Deciding how many tokens to give to the summary versus the buffer. A typical split reserves 20-30% for the summary and 70-80% for recent messages.

### How It Works

New messages are appended to the buffer.

When the buffer exceeds its configured threshold (for example, t tokens), the oldest buffer messages are removed.

Those removed messages get incorporated into the running summary via an LLM summarization call.

The next LLM prompt is assembled as: [system] + [summary of older history] + [recent buffer messages].

This cycle continues. The summary grows and compresses while the buffer always reflects the latest exchanges.

> **The complete progression so far:**
>
> | # | Technique | Mechanism | Gap it leaves |
> |---|---|---|---|
> | 01 | Buffer | Keep everything verbatim | Unbounded cost |
> | 02 | Sliding Window | Keep only last N turns | Loses early context |
> | 03 | Summary Memory | Compress old turns into summary | Recent turns still expensive |
> | **04** | **Summary Buffer** | **Summary of old + token-capped buffer of recent** | **The production hybrid** |
>
> Technique 04 is where the first four techniques converge. It is the pattern most production short-term memory systems actually implement.

### The mental model — a briefing document plus a live notepad

Imagine a financial advisor who starts every client meeting with a one-page briefing document summarising everything from previous sessions, then takes fresh notes during the current meeting. The briefing covers the past. The notepad covers the present. Together they give complete situational awareness.

```
Every API call receives:

  ┌─────────────────────────────────────────────────────────┐
  │ [System: FinCoach persona]                               │
  │ [System: SUMMARY — Chiru earns ₹1,20,000, expenses      │  ← Compressed past
  │  ₹60,000, FD ₹50,000 maturing Q3, risk-averse...]       │
  │ [Turn N-2 verbatim]  }                                   │
  │ [Turn N-1 verbatim]  }  ← token-capped recent buffer    │
  │ [Turn N   verbatim]  }                                   │
  └─────────────────────────────────────────────────────────┘
```

The summary handles permanence. The buffer handles recency. The token budget ensures cost never blows up.

---

### How it differs from Technique 03

| | Technique 03 — Summary Memory | Technique 04 — Summary Buffer |
|---|---|---|
| Trigger | Turn count crosses a threshold | Recent buffer token budget exceeded |
| Recent zone | Fixed turn count kept verbatim | Token-budgeted — any size, capped at a limit |
| Control lever | `max_turns_before_summary` | `max_buffer_tokens` |
| Cost predictability | Turn-based | Token-based — more precise |
| Production fit | Good | Better — tokens are what you are billed on |

The key upgrade in Technique 04 is that the recent buffer is controlled by **tokens**, not turns. A turn where the user pastes a long document costs 500 tokens. A turn where they say "ok" costs 3 tokens. Token-based budgeting reflects actual API cost; turn-based budgeting does not.

---

### Point 1 — Token budget is the correct control unit for production

You are billed in tokens, not turns. A sliding window of 5 turns might cost 200 tokens or 5,000 tokens depending on how verbose the conversation is. A token budget of 2,000 always costs at most 2,000 tokens. Token-based budgeting gives you predictable, auditable API costs.

---

### Point 2 — Overflow triggers compression, not rejection

In Technique 01 (Buffer), hitting the budget raised a `BufferOverflowError` — the caller had to handle it. In Summary Buffer Memory, the overflow is handled internally: the oldest messages in the buffer are automatically compressed into the rolling summary, freeing budget for new messages. No error. No caller action needed.

---

### Point 3 — The summary and buffer are separate zones with separate tokens

The summary lives outside the token budget. It has its own soft cap (set by the summarisation prompt's word limit). The token budget governs only the recent buffer. This separation makes the cost model predictable:

```
Total input tokens per call =
  system_prompt_tokens        (constant ~130)
  + summary_tokens            (soft-capped ~150 words)
  + recent_buffer_tokens      (hard-capped at max_buffer_tokens)
```

You can calculate the worst-case input cost before writing a single line of application code.

---

### Point 4 — This is the pattern LangChain calls `ConversationSummaryBufferMemory`

LangChain's `ConversationSummaryBufferMemory` is an implementation of exactly this pattern. Most production LangChain agents with long sessions use it. By building it from scratch here, you understand every decision that LangChain made under the hood — and you can tune it for your domain rather than accepting defaults.

---

### Point 5 — This is the final short-term memory technique before long-term memory

Techniques 01–04 are all about managing the **context window** — what the model sees in a single API call. After Technique 04, we move to long-term memory (Techniques 06–11) — storing and retrieving facts across sessions. Summary Buffer Memory is the bridge: it is the best within-session memory pattern, and the summary it produces is also the ideal input for the long-term memory layer.

---

## Trade-offs

| | |
|---|---|
| ✅ Bounded, predictable token cost per call | ❌ Two components to tune (summary + buffer) |
| ✅ Early facts preserved via summary | ❌ Summary is still lossy |
| ✅ Recent turns always verbatim | ❌ Adds latency when summarisation fires |
| ✅ Token-precise cost control | ❌ More complex than pure window or summary alone |
| ✅ The production default for short-term memory | ❌ Still no cross-session persistence (→ Technique 06) |

## Production Verdict

> **This is the production default for short-term agent memory. Ship this.**
>
> Summary Buffer Memory is what you build when you stop prototyping and start deploying. It gives you the cost control of a sliding window, the memory preservation of summary memory, and the precision of token-based budgeting. Most mature production agent systems converge on this pattern or a slight variant of it for within-session context management.

---

In [1]:
# Install required packages.
# 'openai'   — OpenAI SDK for GPT-4o API calls.
# 'tiktoken' — GPT-4o's exact tokeniser for precise budget enforcement.
!pip install openai tiktoken --quiet

In [2]:
# --- Standard library ---
import json                              # Session persistence.
import os                                # Environment variables.
import time                              # Rate-limit delays.
from datetime import datetime, timezone  # UTC timestamps, Python 3.12+ safe.
from typing import List, Dict, Optional  # Type hints.
from dataclasses import dataclass, field, asdict  # Data models.

# --- Third-party ---
import openai    # OpenAI SDK.
import tiktoken  # Exact GPT-4o tokeniser.

In [3]:
# --- API Key Setup ---
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    # Load from Colab Secrets vault — safest method.
    print("Key loaded from Colab Secrets.")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    # Fall back to environment variable for local Jupyter.
    print("Key loaded from environment variable.")

assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), \
    "API key missing or invalid. Add OPENAI_API_KEY to Colab Secrets."

client = openai.OpenAI(api_key=OPENAI_API_KEY)
# Single client instance — reused for all conversation and summarisation calls.

MODEL         = "gpt-4o"
# Main model for the FinCoach conversation.

SUMMARY_MODEL = "gpt-4o-mini"
# Cheaper model used exclusively for summarisation.
# Summarisation is compression, not reasoning — mini is sufficient.
# Cost: $0.15/M input tokens vs $2.50/M for gpt-4o — 16x cheaper.

TOKENISER = tiktoken.get_encoding("o200k_base")
# Exact tokeniser for gpt-4o and gpt-4o-mini.
# Token counts here are precise — not approximations.

print(f"Main model    : {MODEL}")
print(f"Summary model : {SUMMARY_MODEL}")

Key loaded from environment variable.
Main model    : gpt-4o
Summary model : gpt-4o-mini


---
## Part 1 — The Message Data Model

Identical across all four techniques. Consistent data model throughout the course.

In [5]:
@dataclass
class Message:
    """A single conversation message — role, content, timestamp."""

    role: str
    # 'user' or 'assistant' — who sent this message.

    content: str
    # The message text.

    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    # UTC timestamp — auto-set at creation. Python 3.12+ compatible.

    def to_api_format(self) -> Dict[str, str]:
        """Return only role and content — the OpenAI API format."""
        return {"role": self.role, "content": self.content}

    def token_count(self) -> int:
        """Count this message's tokens using the exact GPT-4o tokeniser."""
        return len(TOKENISER.encode(self.content))
        # Convenience method — avoids repeating the tokeniser call at the call site.

print("Message dataclass defined.")

Message dataclass defined.


---
## Part 2 — The Summarisation Engine

Same domain-specific prompt and engine as Technique 03. Reused here unchanged — the summarisation component is independent of which memory class uses it.

In [6]:
# Domain-specific summarisation prompt — same as Technique 03.
# This prompt is the most important engineering asset in this technique.
# It explicitly lists every category of fact that must survive compression.
FINCOACH_SUMMARY_PROMPT = """You are summarising a financial advisory conversation for FinCoach.
Your summary will be used as the ONLY memory of earlier turns in the next API call.

CRITICAL — preserve ALL of the following if mentioned:
1. FINANCIAL FACTS: salary, take-home income, expenses, savings rate, assets, liabilities
2. INVESTMENTS: existing investments (FDs, SIPs, stocks, property), values, maturity dates
3. RISK PROFILE: stated risk tolerance (conservative / moderate / aggressive)
4. GOALS: short-term and long-term financial goals
5. CONSTRAINTS: any hard rules the user stated (e.g. never invest in equity)
6. DECISIONS MADE: any investment or financial decisions confirmed by the user
7. ADVICE GIVEN: key recommendations FinCoach made that the user acknowledged
8. PERSONAL CONTEXT: age, family situation, employer, life events

Format rules:
- Third person ("The user...", "FinCoach advised...")
- Single coherent paragraph — no bullet points
- Maximum 150 words — be concise but complete
- Do NOT add facts not in the conversation
"""


def summarise_messages(
    messages_to_compress: List[Message],
    existing_summary: Optional[str] = None
) -> str:
    """
    Compress a list of Message objects into a summary string.
    Incorporates any existing summary for progressive summarisation.

    Args:
        messages_to_compress: Messages to compress — from the buffer's overflow.
        existing_summary:     Prior summary string, or None for first compression.

    Returns:
        Updated summary string incorporating all input.
    """

    turns_text = "\n".join(
        f"{m.role.upper()}: {m.content}" for m in messages_to_compress
    )
    # Format messages as "ROLE: content" lines for the summariser.

    if existing_summary:
        user_content = (
            f"EXISTING SUMMARY (from previous turns):\n{existing_summary}\n\n"
            f"NEW MESSAGES TO INCORPORATE:\n{turns_text}\n\n"
            f"Produce an updated summary that merges BOTH. Do not lose any fact from either."
        )
        # Progressive summarisation: new summary absorbs old summary + new messages.
    else:
        user_content = (
            f"MESSAGES TO SUMMARISE:\n{turns_text}\n\n"
            f"Produce a summary of these messages."
        )
        # First compression cycle — no prior summary to incorporate.

    response = client.chat.completions.create(
        model=SUMMARY_MODEL,
        # gpt-4o-mini: 16x cheaper than gpt-4o for this compression task.
        max_tokens=300,
        # Cap at 300 tokens — our prompt says 150 words, this gives headroom.
        temperature=0.0,
        # Fully deterministic — factual accuracy over creativity.
        messages=[
            {"role": "system", "content": FINCOACH_SUMMARY_PROMPT},
            {"role": "user",   "content": user_content},
        ]
    )

    summary = response.choices[0].message.content.strip()

    original_tokens = sum(m.token_count() for m in messages_to_compress)
    summary_tokens  = len(TOKENISER.encode(summary))

    print(f"    [COMPRESS] {len(messages_to_compress)} msgs ({original_tokens} tokens) "
          f"→ summary ({summary_tokens} tokens) — {original_tokens/max(summary_tokens,1):.1f}x compression")

    return summary


print("Summarisation engine defined.")

Summarisation engine defined.


---
## Part 3 — The SummaryBufferMemory Class

This is the central class of this technique. The key difference from Technique 03:
- Technique 03 triggers compression on **turn count**
- Technique 04 triggers compression on **token budget** — a more precise and production-correct control

The buffer is checked on every `add_message()` call. If adding the new message would exceed `max_buffer_tokens`, the oldest messages in the buffer are compressed first, then the new message is added.

In [7]:
class SummaryBufferMemory:
    """
    Hybrid memory: rolling summary of old messages + token-budgeted buffer of recent ones.

    The token budget governs the recent buffer.
    When the buffer overflows, the oldest messages are compressed into the summary.
    The summary is injected into every API call as a system context block.

    Cost model (fully predictable):
      Per-call input tokens ≈
        system_prompt_tokens   (constant)
      + summary_tokens         (bounded by summary prompt's 150-word cap)
      + recent_buffer_tokens   (hard-capped at max_buffer_tokens)
    """

    # ------------------------------------------------------------------
    # INITIALISATION
    # ------------------------------------------------------------------

    def __init__(
        self,
        session_id: str,
        user_id: str,
        system_prompt: str,
        max_buffer_tokens: int = 1_500,
        # Hard token budget for the recent message buffer.
        # When buffer exceeds this, compression fires automatically.
        # Default 1,500: enough for ~8-10 typical financial advice turns.
        # Production tuning guide:
        #   Short sessions (< 10 turns):  500-1000 tokens
        #   Medium sessions (10-30 turns): 1000-2000 tokens
        #   Long sessions (30+ turns):    1500-3000 tokens
        messages_to_compress_on_overflow: int = 6,
        # When overflow fires, compress this many of the oldest messages.
        # Default 6: compress 3 complete turns (user + assistant = 2 msgs per turn).
        # Must be even to preserve turn pairs — compressing mid-turn causes context issues.
        # Larger value = more aggressive compression = fewer summarisation calls.
    ):
        self.session_id   = session_id
        self.user_id      = user_id
        self.system_prompt = system_prompt

        self.max_buffer_tokens = max_buffer_tokens
        # The hard token budget for self._buffer.
        # The summary lives outside this budget — it has its own soft cap.

        self.messages_to_compress_on_overflow = messages_to_compress_on_overflow
        # How many messages to compress when overflow fires.
        # Always keep this even — compressing an odd number splits a turn pair.

        self._buffer: List[Message] = []
        # The recent message buffer — verbatim, token-budgeted.
        # Grows with each new message. Shrinks when overflow fires.
        # This is what gets sent to the API as the recent context.

        self._summary: Optional[str] = None
        # The rolling summary of all messages that have been compressed out of the buffer.
        # None until the first overflow. Grows via progressive summarisation.
        # Injected into every API call as a system context block once it exists.

        self._archive: List[Message] = []
        # The complete session history — every message ever, never evicted.
        # Not sent to the API. Used for compliance, auditing, and long-term retrieval.

        self._total_turns     = 0  # Complete turns ever (each user+assistant pair = 1).
        self._overflow_count  = 0  # Times the buffer has overflowed and been compressed.
        self.created_at = datetime.now(timezone.utc).isoformat()

        print(f"[SummaryBuffer] Initialised — session: {self.session_id}")
        print(f"  Token budget  : {self.max_buffer_tokens:,} tokens for recent buffer")
        print(f"  On overflow   : compress oldest {self.messages_to_compress_on_overflow} messages")

    # ------------------------------------------------------------------
    # TOKEN COUNTING
    # ------------------------------------------------------------------

    def _buffer_tokens(self) -> int:
        """Count the tokens currently in the recent buffer."""
        return sum(m.token_count() for m in self._buffer)
        # Sum token counts across all messages in the buffer.

    def _summary_tokens(self) -> int:
        """Count the tokens in the current summary (0 if no summary yet)."""
        return len(TOKENISER.encode(self._summary)) if self._summary else 0

    def _system_tokens(self) -> int:
        """Count the tokens in the system prompt (constant overhead)."""
        return len(TOKENISER.encode(self.system_prompt))

    def get_context_breakdown(self) -> Dict[str, int]:
        """
        Break down the next API call's input token cost by zone.
        This is the metric you monitor in production to validate budget compliance.
        """
        buf   = self._buffer_tokens()
        summ  = self._summary_tokens()
        sys   = self._system_tokens()
        return {
            "system":  sys,
            "summary": summ,
            "buffer":  buf,
            "total":   sys + summ + buf,
        }

    # ------------------------------------------------------------------
    # OVERFLOW HANDLER
    # ------------------------------------------------------------------

    def _handle_overflow(self, incoming_tokens: int) -> None:
        """
        Called when adding a new message would push the buffer over budget.
        Compresses the oldest N messages from the buffer into the rolling summary,
        freeing enough budget to accept the new message.

        Args:
            incoming_tokens: Token count of the message about to be added.
                             Used to verify compression freed sufficient space.
        """

        self._overflow_count += 1
        # Track overflow frequency — useful for tuning max_buffer_tokens.
        # If overflow fires every 2-3 turns, your budget is too small.

        print(f"\n[SummaryBuffer] Buffer overflow #{self._overflow_count} — "
              f"buffer at {self._buffer_tokens()} tokens / {self.max_buffer_tokens} budget")

        # Ensure we compress an even number of messages to avoid splitting turn pairs.
        compress_count = min(
            self.messages_to_compress_on_overflow,  # Desired compression count.
            len(self._buffer)                        # Cannot compress more than exists.
        )
        # Clamp to even number: always compress complete turn pairs.
        if compress_count % 2 != 0:
            compress_count -= 1
            # If odd, drop by 1 to make it even — preserves turn pair integrity.

        if compress_count == 0:
            # No messages to compress — buffer has fewer messages than the compress target.
            # This should not happen with correct settings but guard defensively.
            print("  WARNING: nothing to compress — buffer too small for compression target")
            return

        messages_to_compress = self._buffer[:compress_count]
        # Slice the OLDEST messages from the front of the buffer.
        # These are the messages with the lowest recency — compress first.

        remaining_buffer = self._buffer[compress_count:]
        # Everything after the compressed messages — stays verbatim in the buffer.

        print(f"  Compressing {compress_count} messages "
              f"({sum(m.token_count() for m in messages_to_compress)} tokens)")

        # Run the summarisation — incorporates any existing summary.
        self._summary = summarise_messages(
            messages_to_compress=messages_to_compress,
            existing_summary=self._summary,
            # Pass the current summary for progressive summarisation.
            # If this is the first overflow, existing_summary=None → fresh summary.
            # If this is a subsequent overflow, it builds on the prior summary.
        )

        self._buffer = remaining_buffer
        # Replace the buffer with only the non-compressed portion.
        # The compressed messages now exist only in:
        #   self._summary  (compressed form)
        #   self._archive  (full verbatim form)

        freed_tokens = sum(m.token_count() for m in messages_to_compress)
        print(f"  Buffer after compression: {self._buffer_tokens()} tokens "
              f"(freed {freed_tokens} tokens)")

    # ------------------------------------------------------------------
    # CORE OPERATIONS
    # ------------------------------------------------------------------

    def add_message(self, role: str, content: str) -> None:
        """
        Add a message to the buffer.
        If adding the message would exceed the token budget,
        overflow handling fires first to free space.
        """

        if role not in ("user", "assistant"):
            raise ValueError(f"Invalid role '{role}'. Use 'user' or 'assistant'.")

        new_message = Message(role=role, content=content)
        # Create the message — timestamp auto-set to UTC now.

        incoming_tokens = new_message.token_count()
        # Count the new message's tokens before deciding whether to compress.

        projected_buffer_tokens = self._buffer_tokens() + incoming_tokens
        # What the buffer token count would be AFTER adding this message.

        if projected_buffer_tokens > self.max_buffer_tokens:
            # Adding this message would exceed the budget — handle overflow first.
            # This compresses the oldest messages into the summary, freeing space.
            self._handle_overflow(incoming_tokens=incoming_tokens)
            # After compression, the buffer has more free budget.
            # We then proceed to add the message normally below.

        self._buffer.append(new_message)
        # Add the new message to the (now possibly compressed) buffer.

        self._archive.append(new_message)
        # Always add to the full archive — it never loses messages.

        if role == "assistant":
            self._total_turns += 1
            # Count completed turns — each assistant message marks one complete turn.

    def get_messages_for_api(self) -> List[Dict[str, str]]:
        """
        Build the complete message list for the OpenAI API.

        Structure:
          [system: FinCoach persona]
          [system: rolling summary]  ← injected only when summary exists
          [recent buffer messages verbatim]

        The model sees: who it is + what happened before + what's happening now.
        """

        messages = []

        messages.append({"role": "system", "content": self.system_prompt})
        # 1. FinCoach's persona — always first.

        if self._summary:
            summary_block = (
                f"CONVERSATION MEMORY (earlier turns, compressed):\n"
                f"{self._summary}\n"
                f"The current conversation continues below."
            )
            messages.append({"role": "system", "content": summary_block})
            # 2. Summary injected as a second system message.
            # Marked as 'compressed' so the model understands it is a digest,
            # not a verbatim record — this prevents the model from treating
            # the summary as if it contains exact quotes.

        for msg in self._buffer:
            messages.append(msg.to_api_format())
            # 3. Recent buffer messages — verbatim, in chronological order.
            # These are always within the token budget.

        return messages

    # ------------------------------------------------------------------
    # PERSISTENCE
    # ------------------------------------------------------------------

    def persist(self, filepath: str) -> None:
        """Save the complete session state to a JSON file."""

        ctx = self.get_context_breakdown()

        record = {
            "schema_version": "1.0",
            "technique": "summary_buffer_memory",
            "session_id": self.session_id,
            "user_id": self.user_id,
            "model": MODEL,
            "summary_model": SUMMARY_MODEL,
            "created_at": self.created_at,
            "saved_at": datetime.now(timezone.utc).isoformat(),
            "config": {
                "max_buffer_tokens": self.max_buffer_tokens,
                "messages_to_compress_on_overflow": self.messages_to_compress_on_overflow,
            },
            "stats": {
                "total_turns": self._total_turns,
                "overflow_count": self._overflow_count,
                # How many times overflow fired — useful for tuning the budget.
                "buffer_messages": len(self._buffer),
                "archive_messages": len(self._archive),
                "context_tokens": ctx,
            },
            "summary": self._summary,
            # The rolling summary — the most important state to restore.
            "buffer": [asdict(m) for m in self._buffer],
            # The recent buffer — verbatim messages in the active window.
            "archive": [asdict(m) for m in self._archive],
            # The full session history — for compliance and retrieval.
        }

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)

        print(f"[SummaryBuffer] Persisted — {self._total_turns} turns, "
              f"{self._overflow_count} overflow(s), "
              f"{ctx['total']} context tokens")

    @classmethod
    def load(cls, filepath: str) -> "SummaryBufferMemory":
        """Restore SummaryBufferMemory from a saved JSON file."""

        with open(filepath, "r", encoding="utf-8") as f:
            record = json.load(f)

        if record.get("schema_version") != "1.0":
            raise ValueError(f"Unknown schema version: {record.get('schema_version')}")

        mem = cls(
            session_id=record["session_id"],
            user_id=record["user_id"],
            system_prompt="[LOADED — set system_prompt after loading]",
            max_buffer_tokens=record["config"]["max_buffer_tokens"],
            messages_to_compress_on_overflow=record["config"]["messages_to_compress_on_overflow"],
        )

        mem._summary = record["summary"]
        # Restore the summary — the critical compressed state.

        def restore_messages(msg_list):
            return [
                Message(role=m["role"], content=m["content"], timestamp=m["timestamp"])
                for m in msg_list
            ]

        mem._buffer  = restore_messages(record["buffer"])
        mem._archive = restore_messages(record["archive"])
        mem._total_turns    = record["stats"]["total_turns"]
        mem._overflow_count = record["stats"]["overflow_count"]
        mem.created_at      = record["created_at"]

        print(f"[SummaryBuffer] Loaded — {mem._total_turns} turns, "
              f"{mem._overflow_count} overflow(s)")
        if mem._summary:
            print(f"  Summary: \"{mem._summary[:80]}...\"")

        return mem

    # ------------------------------------------------------------------
    # DIAGNOSTICS
    # ------------------------------------------------------------------

    def print_stats(self) -> None:
        """Print a full breakdown of memory state and token usage."""
        ctx = self.get_context_breakdown()

        print(f"\n{'='*62}")
        print(f"  Summary Buffer Stats — Session: {self.session_id}")
        print(f"{'='*62}")
        print(f"  Total turns ever     : {self._total_turns}")
        print(f"  Overflow fires       : {self._overflow_count}")
        print(f"  Buffer messages      : {len(self._buffer)} "
              f"({len(self._buffer)//2} turns)")
        print(f"  Archive messages     : {len(self._archive)} (full history)")
        print()
        print(f"  Token breakdown — next API call:")
        print(f"    System prompt  : {ctx['system']:>6,} tokens  (constant)")
        print(f"    Summary block  : {ctx['summary']:>6,} tokens  (compressed past)")
        print(f"    Recent buffer  : {ctx['buffer']:>6,} / {self.max_buffer_tokens:,} tokens  (verbatim)")
        print(f"    ────────────────────────────")
        print(f"    TOTAL input    : {ctx['total']:>6,} tokens")
        print()

        # Buffer utilisation bar.
        util = min(ctx['buffer'] / self.max_buffer_tokens, 1.0)
        bar  = "█" * int(30 * util) + "░" * (30 - int(30 * util))
        print(f"  Buffer fill : [{bar}] {util:.0%} of {self.max_buffer_tokens} token budget")

        if self._summary:
            print(f"\n  Rolling summary:")
            print(f"  {self._summary}")
        else:
            print(f"\n  No summary yet — buffer has not overflowed.")
        print(f"{'='*62}\n")


print("SummaryBufferMemory class defined.")

SummaryBufferMemory class defined.


---
## Part 4 — The FinCoach Agent

In [8]:
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Your principles:
- Always personalise advice using information the user has shared in this conversation.
- Be specific with numbers when the user has provided their financial details.
- Flag when you are making assumptions due to missing information.
- Keep responses concise — 3 to 5 sentences unless the user asks for detail.
- Never provide specific buy/sell recommendations on individual stocks.
- Always recommend consulting a SEBI-registered advisor for major financial decisions.

Use ALL available context — the conversation memory block and recent messages — for personalised advice."""


def chat(
    memory: SummaryBufferMemory,
    user_message: str,
    verbose: bool = True
) -> str:
    """
    Send one user message to FinCoach. Return the assistant's reply.
    Overflow (if triggered) fires inside add_message() — transparent to the caller.
    """

    # STEP 1 — Add the user message.
    # If adding this message overflows the buffer, _handle_overflow() fires
    # automatically before the message is added. No caller action needed.
    memory.add_message(role="user", content=user_message)

    # STEP 2 — Call the API with summary + buffer context.
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=1024,
        temperature=0.7,
        messages=memory.get_messages_for_api(),
        # Returns: [system: persona] + [system: summary?] + [buffer verbatim]
        # This is the hybrid context that gives FinCoach both past and present.
    )

    # STEP 3 — Extract the reply.
    assistant_reply = response.choices[0].message.content

    # STEP 4 — Add the reply to the buffer.
    # If the reply itself overflows the buffer, overflow fires again here.
    # Long responses (detailed financial plans) can trigger this.
    memory.add_message(role="assistant", content=assistant_reply)

    if verbose:
        ctx = memory.get_context_breakdown()
        print(f"[Turn {memory._total_turns}] "
              f"API input: {response.usage.prompt_tokens} tokens | "
              f"Buffer: {ctx['buffer']}/{memory.max_buffer_tokens} | "
              f"Summary: {ctx['summary']} tokens | "
              f"Overflows: {memory._overflow_count}")

    return assistant_reply


print("FinCoach chat() function defined.")

FinCoach chat() function defined.


---
## Part 5 — Demo: The Production Conversation

We run a 10-turn FinCoach session with a tight token budget to force overflow during the demo. Watch:
- When overflow fires (buffer tokens approach the budget)
- The summary being built from compressed messages
- The late turns correctly referencing early facts via the summary
- The buffer token count staying bounded throughout

In [9]:
sb_memory = SummaryBufferMemory(
    session_id="session_sb_demo_001",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_buffer_tokens=800,
    # Tight budget: 800 tokens forces overflow after ~4-5 turns.
    # In production: use 1500-2500 for FinCoach.
    # Small budget here so overflow is visible during the demo.
    messages_to_compress_on_overflow=4,
    # Compress the oldest 4 messages (2 turns) when overflow fires.
)

demo_turns = [
    "Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.",
    # Turn 1: critical fact — salary.
    "My monthly expenses are ₹60,000. I'm 32 years old and risk-averse.",
    # Turn 2: expenses, age, risk profile.
    "I have an FD of ₹50,000 maturing in 3 months and no other investments.",
    # Turn 3: existing assets.
    "My financial goal is to retire by 55 with a comfortable corpus.",
    # Turn 4: long-term goal. Overflow likely fires after this turn.
    "What is the difference between PPF and NPS for retirement planning?",
    # Turn 5: generic question — tests whether Turn 1-4 facts survive in summary.
    "Based on my salary and risk profile, which one would you recommend?",
    # Turn 6: requires salary and risk profile from summary.
    "How much should I invest in NPS each month to retire comfortably by 55?",
    # Turn 7: requires age (32), retirement goal (55), and salary.
    "Should I also keep some allocation in debt mutual funds?",
    # Turn 8: continuation.
    "What tax benefits do I get under Section 80C if I invest in NPS and ELSS?",
    # Turn 9: tax question.
    "Can you give me a complete action plan based on everything we have discussed?",
    # Turn 10: synthesis — requires ALL facts from the entire session.
    # This is the ultimate test: can the model produce a complete plan
    # using facts compressed into the summary in early turns?
]

print("SUMMARY BUFFER MEMORY — 10-Turn FinCoach Demo")
print("Budget: 800 tokens | Compress: 4 messages per overflow")
print("=" * 65)

for i, user_message in enumerate(demo_turns, start=1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_message}")
    response = chat(memory=sb_memory, user_message=user_message, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

print("\n" + "=" * 65)
sb_memory.print_stats()

[SummaryBuffer] Initialised — session: session_sb_demo_001
  Token budget  : 800 tokens for recent buffer
  On overflow   : compress oldest 4 messages
SUMMARY BUFFER MEMORY — 10-Turn FinCoach Demo
Budget: 800 tokens | Compress: 4 messages per overflow

--- Turn 1 ---
User: Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.
[Turn 1] API input: 165 tokens | Buffer: 162/800 | Summary: 0 tokens | Overflows: 0
FinCoach: Hi Chiru! With a monthly take-home salary of ₹1,20,000, you have a solid base to work from for saving, investing, and budgeting. To start, it's generally advisable to allocate around 20% of your income to savings and investments, which would be ₹24,000 per month. You might also consider setting aside 50% for essential expenses (₹60,000) and the remaining 30% (₹36,000) for discretionary spending. This is a guideline known as the 50/30/20 rule, which you can adjust to better fit your personal financial goals and obligations. Let me know if you need assistance with specif

---
## Part 6 — Budget Tuning: Finding the Right `max_buffer_tokens`

In [10]:
def analyse_buffer_budget(
    turns: List[str],
    budget_options: List[int],
    avg_response_tokens: int = 100
) -> None:
    """
    Simulate how often overflow fires for different token budget settings.
    No API calls — pure token arithmetic.
    Run this BEFORE deploying to choose the right max_buffer_tokens.

    Args:
        turns:               Simulated user messages.
        budget_options:      List of token budgets to evaluate.
        avg_response_tokens: Estimated average assistant response length.
    """

    print("BUFFER BUDGET ANALYSIS — overflow frequency per budget setting")
    print("=" * 65)
    print(f"Session: {len(turns)} turns, avg response: {avg_response_tokens} tokens")
    print()
    print(f"{'Budget':>10} | {'Overflows':>10} | {'Max buffer':>12} | {'Recommended?':>14}")
    print("-" * 55)

    for budget in budget_options:
        # Simulate the buffer filling up across the session.
        simulated_buffer_tokens = 0
        overflow_count = 0
        max_buffer_seen = 0

        for user_msg in turns:
            user_tokens = len(TOKENISER.encode(user_msg))

            # Simulate adding user message — check overflow.
            if simulated_buffer_tokens + user_tokens > budget:
                overflow_count += 1
                # Simulate compression: remove oldest 4 messages' worth of tokens.
                # (Approximation: assume avg message is 60 tokens)
                simulated_buffer_tokens = max(0, simulated_buffer_tokens - 4 * 60)

            simulated_buffer_tokens += user_tokens

            # Simulate adding assistant response.
            if simulated_buffer_tokens + avg_response_tokens > budget:
                overflow_count += 1
                simulated_buffer_tokens = max(0, simulated_buffer_tokens - 4 * 60)

            simulated_buffer_tokens += avg_response_tokens
            max_buffer_seen = max(max_buffer_seen, simulated_buffer_tokens)

        # Determine recommendation.
        if overflow_count == 0:
            rec = "✅ No overflow"
        elif overflow_count <= 2:
            rec = "✅ Good balance"
        elif overflow_count <= 5:
            rec = "⚠ Frequent"
        else:
            rec = "❌ Too small"

        print(f"{budget:>10,} | {overflow_count:>10} | "
              f"{max_buffer_seen:>12,} | {rec:>14}")

    print()
    print("Rule of thumb:")
    print("  0-1 overflows   → budget may be too large (paying for buffer space you don't need)")
    print("  2-3 overflows   → good balance: facts preserved, cost controlled")
    print("  5+ overflows    → budget too small: too many compression cycles, info loss risk")


# Representative FinCoach session turns for analysis.
analysis_turns = [
    "Hi, my monthly salary is ₹1,20,000 and expenses are ₹60,000.",
    "I'm 32, risk-averse, and want to retire at 55.",
    "I have an FD of ₹50,000 maturing next quarter.",
    "What is the difference between PPF, NPS, and ELSS?",
    "Which of these suits a conservative investor like me?",
    "How much should I invest monthly to build a ₹2 crore corpus by 55?",
    "Should I invest my FD maturity amount in a lump sum SIP?",
    "What are the tax implications of NPS under the new tax regime?",
    "Should I open a PPF account given the 15-year lock-in?",
    "Give me a complete monthly investment breakdown.",
]

analyse_buffer_budget(
    turns=analysis_turns,
    budget_options=[400, 600, 800, 1_000, 1_500, 2_000, 3_000],
)

BUFFER BUDGET ANALYSIS — overflow frequency per budget setting
Session: 10 turns, avg response: 100 tokens

    Budget |  Overflows |   Max buffer |   Recommended?
-------------------------------------------------------
       400 |          4 |          350 |     ⚠ Frequent
       600 |          3 |          575 |     ⚠ Frequent
       800 |          2 |          794 | ✅ Good balance
     1,000 |          1 |          919 | ✅ Good balance
     1,500 |          0 |        1,142 |  ✅ No overflow
     2,000 |          0 |        1,142 |  ✅ No overflow
     3,000 |          0 |        1,142 |  ✅ No overflow

Rule of thumb:
  0-1 overflows   → budget may be too large (paying for buffer space you don't need)
  2-3 overflows   → good balance: facts preserved, cost controlled
  5+ overflows    → budget too small: too many compression cycles, info loss risk


---
## Part 7 — Session Persistence

In [11]:
SESSION_FILE = "/tmp/fincoach_sb_session_demo_001.json"

sb_memory.persist(SESSION_FILE)
# Saves: summary, buffer, archive, config, stats.

print("\n--- Simulating service restart ---\n")

restored = SummaryBufferMemory.load(SESSION_FILE)
restored.system_prompt = FINCOACH_SYSTEM_PROMPT

print("\nResuming the session after restore:")
follow_up = "Given everything we discussed, what is my exact monthly surplus and what should I do with it?"
print(f"User: {follow_up}")

response = chat(memory=restored, user_message=follow_up, verbose=True)
# The restored memory has the rolling summary (salary, expenses, FD, risk profile, goals)
# AND the recent buffer (last few turns verbatim).
# FinCoach should answer with ₹60,000 surplus without being re-told.
print(f"FinCoach: {response}")

[SummaryBuffer] Persisted — 10 turns, 4 overflow(s), 1044 context tokens

--- Simulating service restart ---

[SummaryBuffer] Initialised — session: session_sb_demo_001
  Token budget  : 800 tokens for recent buffer
  On overflow   : compress oldest 4 messages
[SummaryBuffer] Loaded — 10 turns, 4 overflow(s)
  Summary: "The user, Chiru, is 32 years old and has a monthly take-home salary of ₹1,20,000..."

Resuming the session after restore:
User: Given everything we discussed, what is my exact monthly surplus and what should I do with it?

[SummaryBuffer] Buffer overflow #5 — buffer at 670 tokens / 800 budget
  Compressing 4 messages (651 tokens)
    [COMPRESS] 4 msgs (651 tokens) → summary (235 tokens) — 2.8x compression
  Buffer after compression: 19 tokens (freed 651 tokens)
[Turn 11] API input: 1111 tokens | Buffer: 258/800 | Summary: 235 tokens | Overflows: 5
FinCoach: Based on the information you've shared, your monthly take-home salary is ₹1,20,000 and your monthly expenses are ₹

---
## Part 8 — The Complete Four-Way Comparison

In [12]:
# Illustrative — no API calls.
# The definitive comparison of all four short-term memory techniques.

print("FOUR-WAY COMPARISON — What the API receives at Turn 10 of a FinCoach session")
print("=" * 70)

print("""
01 — CONVERSATION BUFFER
────────────────────────
  [system: persona]
  [T1][T2][T3][T4][T5][T6][T7][T8][T9][T10]  ← ALL 10 turns verbatim

  ✅ Nothing lost
  ❌ Cost grows every turn — T10 costs 10x T1
  ❌ Context overflow risk at long sessions
  ❌ No cross-session persistence
""")

print("""
02 — SLIDING WINDOW (size=5 turns)
───────────────────────────────────
  [system: persona]
  [T6][T7][T8][T9][T10]  ← last 5 turns verbatim, T1-T5 DROPPED

  ✅ Fixed cost — T100 costs same as T5
  ❌ T1-T5 are gone — salary, risk profile, goals LOST
  ❌ Context echo is the only 'memory' of early facts
""")

print("""
03 — SUMMARY MEMORY (keep 3 turns verbatim)
─────────────────────────────────────────────
  [system: persona]
  [system: SUMMARY of T1-T7 — ~120 tokens]
  [T8][T9][T10]  ← last 3 turns verbatim

  ✅ Early facts preserved in summary
  ✅ Bounded cost
  ⚠  Recent zone may still be large if turns are verbose
  ⚠  Trigger is turn-count based — not aligned to actual token cost
""")

print("""
04 — SUMMARY BUFFER (max_buffer=1500 tokens) ← THE PRODUCTION DEFAULT
────────────────────────────────────────────────────────────────────────
  [system: persona]                           ~130 tokens
  [system: SUMMARY of T1-T6 — ~120 tokens]    ~120 tokens
  [T7][T8][T9][T10]  ← recent buffer          ~1,200 tokens
  ─────────────────────────────────────────────────────────
  TOTAL:                                       ~1,450 tokens  ← BOUNDED

  ✅ Early facts preserved in summary
  ✅ Recent turns verbatim for conversational coherence
  ✅ Token-precise budget — what you're billed on
  ✅ Cost is predictable: sys + summary_cap + max_buffer_tokens
  ✅ Overflow handled automatically — no caller error handling needed
  ⚠  Summary is still lossy — prompt engineering matters
  ⚠  Two components to tune (summary cap + buffer budget)
""")

print("=" * 70)
print("Techniques 01–04 are ALL about managing the context window.")
print("Technique 06 onwards adds cross-session persistence — long-term memory.")
print("Summary Buffer is the bridge: its summary is the input for long-term storage.")

FOUR-WAY COMPARISON — What the API receives at Turn 10 of a FinCoach session

01 — CONVERSATION BUFFER
────────────────────────
  [system: persona]
  [T1][T2][T3][T4][T5][T6][T7][T8][T9][T10]  ← ALL 10 turns verbatim

  ✅ Nothing lost
  ❌ Cost grows every turn — T10 costs 10x T1
  ❌ Context overflow risk at long sessions
  ❌ No cross-session persistence


02 — SLIDING WINDOW (size=5 turns)
───────────────────────────────────
  [system: persona]
  [T6][T7][T8][T9][T10]  ← last 5 turns verbatim, T1-T5 DROPPED

  ✅ Fixed cost — T100 costs same as T5
  ❌ T1-T5 are gone — salary, risk profile, goals LOST
  ❌ Context echo is the only 'memory' of early facts


03 — SUMMARY MEMORY (keep 3 turns verbatim)
─────────────────────────────────────────────
  [system: persona]
  [system: SUMMARY of T1-T7 — ~120 tokens]
  [T8][T9][T10]  ← last 3 turns verbatim

  ✅ Early facts preserved in summary
  ✅ Bounded cost
  ⚠  Recent zone may still be large if turns are verbose
  ⚠  Trigger is turn-count based

---
## Key Takeaways

**What you built:** A `SummaryBufferMemory` class with token-based overflow detection, automatic progressive summarisation, bounded context cost, session persistence, and a budget tuning utility.

---

### The complete short-term memory picture

| | 01 Buffer | 02 Window | 03 Summary | 04 Summary Buffer |
|---|---|---|---|---|
| Early facts kept | ✅ All | ❌ Evicted | ✅ Compressed | ✅ Compressed |
| Cost bounded | ❌ | ✅ | ✅ | ✅ |
| Control unit | — | Turns | Turns | **Tokens** |
| Overflow handling | ❌ Error | Auto-evict | Auto-compress | Auto-compress |
| Production default | ❌ | ❌ | Partial | **Yes** |

---

### The three things to remember

1. **Token budget is the correct control lever.** Turns are a proxy. Tokens are what you pay for. Token-based budgeting gives you predictable, auditable cost per API call — regardless of how verbose the conversation is.

2. **Overflow fires automatically — no caller error handling.** This is the user experience upgrade over Technique 01's `BufferOverflowError`. The caller just calls `add_message()` — the memory class handles the rest. In production, you still want to log and monitor overflow events, but the agent never fails because of them.

3. **The summary produced here is the input for long-term memory.** When Technique 06 (Vector Store) and Technique 07 (Entity Memory) store facts across sessions, they pull from the archive and the summary — not from raw API call logs. Summary Buffer Memory produces exactly the right format for that hand-off.

---

### What comes next — crossing the session boundary

All four techniques so far manage memory **within a single session**. When the session ends, everything resets — unless you persist it. Techniques 06–11 add **long-term memory** that survives across sessions: vector stores for semantic retrieval, entity memory for structured facts, episodic memory for timestamped events, and semantic memory for distilled truths. Summary Buffer Memory is the foundation they all build on.

